# Ripopt.jl — JuMP Tutorial

This notebook demonstrates how to use [ripopt](https://github.com/jkitchin/ripopt) from Julia via [JuMP](https://jump.dev/).

## Setup

First, build the ripopt shared library:
```bash
cd /path/to/ripopt
cargo build --release
```

Then set the library path and load packages:

In [ ]:
# Point to the ripopt shared library
ENV["RIPOPT_LIBRARY_PATH"] = joinpath(@__DIR__, "..", "..", "target", "release")

using JuMP
using Ripopt

## Example 1: Rosenbrock Function (Unconstrained)

The classic test for nonlinear optimizers:

$$\min_{x} \quad 100(x_2 - x_1^2)^2 + (1 - x_1)^2$$

The global minimum is at $x^* = (1, 1)$ with $f(x^*) = 0$.

In [ ]:
model = Model(Ripopt.Optimizer)
set_silent(model)

@variable(model, x[1:2])
set_start_value(x[1], -1.2)
set_start_value(x[2],  1.0)

@NLobjective(model, Min, 100*(x[2] - x[1]^2)^2 + (1 - x[1])^2)

optimize!(model)

println("Status:    ", termination_status(model))
println("Objective: ", objective_value(model))
println("Solution:  x = ", value.(x))

## Example 2: HS071 (Constrained NLP)

The Hock-Schittkowski problem #71 is a standard test for constrained nonlinear solvers:

$$\min_{x} \quad x_1 x_4 (x_1 + x_2 + x_3) + x_3$$

subject to:
$$x_1 x_2 x_3 x_4 \geq 25$$
$$x_1^2 + x_2^2 + x_3^2 + x_4^2 = 40$$
$$1 \leq x_i \leq 5, \quad i = 1, \ldots, 4$$

Expected solution: $x^* \approx (1.0, 4.743, 3.821, 1.379)$, $f(x^*) \approx 17.014$.

In [ ]:
model = Model(Ripopt.Optimizer)
set_silent(model)

@variable(model, 1 <= x[1:4] <= 5)
set_start_value(x[1], 1.0)
set_start_value(x[2], 5.0)
set_start_value(x[3], 5.0)
set_start_value(x[4], 1.0)

@NLobjective(model, Min, x[1]*x[4]*(x[1]+x[2]+x[3]) + x[3])
@NLconstraint(model, x[1]*x[2]*x[3]*x[4] >= 25)
@NLconstraint(model, x[1]^2 + x[2]^2 + x[3]^2 + x[4]^2 == 40)

optimize!(model)

println("Status:    ", termination_status(model))
println("Objective: ", round(objective_value(model), digits=4))
println("Solution:  x = ", round.(value.(x), digits=4))
println("Solve time: ", round(solve_time(model), sigdigits=3), " s")

## Example 3: Solver Options

Ripopt supports the same option names as Ipopt. You can set them via `set_optimizer_attribute`:

In [ ]:
model = Model(Ripopt.Optimizer)

# Silence output
set_silent(model)

# Solver-specific options
set_optimizer_attribute(model, "tol", 1e-10)           # tighter tolerance
set_optimizer_attribute(model, "max_iter", 500)         # iteration limit
set_optimizer_attribute(model, "mu_strategy", "adaptive") # barrier strategy

# Time limit (standard JuMP attribute)
set_time_limit_sec(model, 60.0)

@variable(model, x[1:2])
set_start_value(x[1], -1.2)
set_start_value(x[2],  1.0)
@NLobjective(model, Min, 100*(x[2] - x[1]^2)^2 + (1 - x[1])^2)

optimize!(model)

println("Status:    ", termination_status(model))
println("Objective: ", objective_value(model))
println("Solution:  ", value.(x))

## Example 4: Maximization

Ripopt handles `Max` sense by internally negating the objective:

$$\max_{x} \quad -(x - 3)^2 + 5$$

Solution: $x^* = 3$, $f(x^*) = 5$.

In [ ]:
model = Model(Ripopt.Optimizer)
set_silent(model)

@variable(model, x)
set_start_value(x, 0.0)

@NLobjective(model, Max, -(x - 3)^2 + 5)

optimize!(model)

println("Status:    ", termination_status(model))
println("Objective: ", round(objective_value(model), digits=6))
println("Solution:  x = ", round(value(x), digits=6))

## Example 5: Comparing with Ipopt

Since Ripopt's interface mirrors Ipopt, switching between them is trivial.
Just change the optimizer:

```julia
# With Ripopt
model = Model(Ripopt.Optimizer)

# With Ipopt (if installed)
# using Ipopt
# model = Model(Ipopt.Optimizer)
```

The rest of the model definition stays identical. This makes it easy to
benchmark Ripopt against Ipopt on your problems.

## Example 6: Economic Dispatch (A Practical Problem)

Minimize the total generation cost of 3 power plants subject to
demand balance and generation limits:

$$\min \sum_{i=1}^{3} (a_i + b_i P_i + c_i P_i^2)$$

subject to:
$$\sum_{i=1}^{3} P_i = P_{\text{demand}}$$
$$P_i^{\min} \leq P_i \leq P_i^{\max}}$$

In [ ]:
model = Model(Ripopt.Optimizer)
set_silent(model)

# Generator cost coefficients: cost_i = a_i + b_i*P_i + c_i*P_i^2
a = [500.0, 300.0, 100.0]
b = [5.3,   5.5,   6.0]
c = [0.004, 0.006, 0.009]

# Generation limits (MW)
P_min = [100.0, 50.0,  80.0]
P_max = [600.0, 400.0, 500.0]

# Demand
P_demand = 850.0

@variable(model, P_min[i] <= P[i=1:3] <= P_max[i])
for i in 1:3
    set_start_value(P[i], P_demand / 3)
end

@NLobjective(model, Min,
    (a[1] + b[1]*P[1] + c[1]*P[1]^2) +
    (a[2] + b[2]*P[2] + c[2]*P[2]^2) +
    (a[3] + b[3]*P[3] + c[3]*P[3]^2)
)

# Power balance
@constraint(model, P[1] + P[2] + P[3] == P_demand)

optimize!(model)

println("Status:       ", termination_status(model))
println("Total cost:   \$", round(objective_value(model), digits=2))
for i in 1:3
    println("  Generator $i: ", round(value(P[i]), digits=2), " MW")
end